<a href="https://colab.research.google.com/github/NobunagaK/online-retail-top10-recommendation/blob/docs%2Ftransaction-records/docs/technical/transaction_records.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Technical reference · Retail transaction records and baskets

The recommender builds purchase histories from invoice lines. This notebook inspects invoice `536365` for customer `17850` in the original Excel workbook and distinguishes line count, distinct products, units purchased, and spend.

In Colab, choose a CPU runtime and run the code cells from top to bottom using ▶. Each calculation reads the original data.

The workbook comes from D. Chen’s *Online Retail* dataset (2015), licensed under CC BY 4.0. Amounts are in pounds sterling. [Dataset](https://doi.org/10.24432/C5BW33)

# 技术文档 · 原始交易明细与购物篮

推荐系统从客户的购买历史中提取商品。本文件记录从原始 Excel 核对一张发票的方法，并区分明细行数、商品种数、购买件数和金额。示例使用客户 `17850` 的发票 `536365`。

在 Colab 选择 CPU，按顺序点击各代码单元格左侧的 ▶，即可用原始数据复核文中的数字。

工作簿来自 D. Chen（2015）的 UCI Online Retail 数据集，许可为 CC BY 4.0。金额单位为英镑。[数据来源](https://doi.org/10.24432/C5BW33)

## Locate the workbook and grant read access

Check that your project folder in Google Drive contains `data/raw/Online Retail.xlsx`. If the folder appears under “Shared with me”, add a shortcut in “My Drive”. Run the next cell in Colab and grant access to mount Drive.

The code checks the project’s default path first. If prompted, find the Excel file under `drive` in Colab’s Files pane, choose “Copy path”, paste the path into the input box, and press Enter. Local Jupyter runs look for the workbook in the repository.

## 工作簿位置与读取权限

先在 Google Drive 里确认项目文件夹下有 `data/raw/Online Retail.xlsx`。如果文件夹在「与我共享」，可以把它作为快捷方式添加到「我的云端硬盘」。在 Colab 运行下一格，按提示授权挂载 Drive。

代码会先检查项目约定的路径。如果出现输入框，打开 Colab 左栏的「文件」，在 `drive` 下找到 Excel 文件，从它的菜单选择「复制路径」，粘贴到输入框并按回车。输入框接收文件路径。使用本地 Jupyter 时，代码会查找仓库内的数据文件。

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    drive.mount("/content/drive")
    workbook = Path("/content/drive/MyDrive/CS5481_Project2_OnlineRetail/data/raw/Online Retail.xlsx")
    if not workbook.is_file():
        workbook = Path(input("Paste the Excel path from Colab Files / 请粘贴从 Colab「文件」面板复制的 Excel 路径：").strip())
else:
    current_dir = Path.cwd()
    for directory in (current_dir, *current_dir.parents):
        workbook = directory / "data/raw/Online Retail.xlsx"
        if workbook.is_file():
            break

if not workbook.is_file():
    raise FileNotFoundError("Workbook missing. Check the filename, path, and read access, then rerun this cell. / 没有找到工作簿。请核对文件名、路径和当前账号的读取权限，再运行本格。")
print("Workbook / 本次读取的文件：", workbook.resolve())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Workbook / 本次读取的文件： /content/drive/MyDrive/CS5481_Project2_OnlineRetail/data/raw/Online Retail.xlsx


## Verify the workbook version and schema

A SHA-256 digest identifies the contents of a file. Two files can share a name but contain different data. The first block compares the workbook digest with the value in the research plan. After it matches, the second block reads the `Online Retail` sheet and treats the three identifier columns as text, preserving letters in product codes. Loading over 540,000 rows can take a moment.

## 数据版本与表结构

SHA-256 是根据文件全部内容算出的校验值。同名文件也可能有不同内容。第一段把实际值和研究计划中的值比较，匹配后才继续。第二段读取名为 `Online Retail` 的工作表，把三个编号列按文本保存，以便保留商品编码里的字母。读取 54 万多行可能需要稍候。

In [8]:
from hashlib import sha256
import pandas as pd

expected_hash = "43465a06f2ccf7c8b5bd2892bc7defb52f97487934fe93b16ae4c3936424676d"
actual_hash = sha256(workbook.read_bytes()).hexdigest()
print("SHA-256 / 文件校验值：", actual_hash)
if actual_hash != expected_hash:
    raise ValueError("Workbook digest mismatch. Select the original file and rerun from the first cell. / 当前文件与研究计划记录的版本不同。请选择未经编辑的原始工作簿，再从第一步运行。")

identifiers = {"InvoiceNo": "string", "StockCode": "string", "CustomerID": "string"}
sales = pd.read_excel(workbook, sheet_name="Online Retail", dtype=identifiers)
print(f"Rows / 交易明细行数：{len(sales):,}   Columns / 列数：{len(sales.columns)}")
print("Fields / 字段：", ", ".join(sales.columns))
assert sales.shape == (541909, 8), "Check the worksheet dimensions. / 请检查工作表的行数和列数。"

SHA-256 / 文件校验值： 43465a06f2ccf7c8b5bd2892bc7defb52f97487934fe93b16ae4c3936424676d
Rows / 交易明细行数：541,909   Columns / 列数：8
Fields / 字段： InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country


## Transaction fields

Each row records the quantity and unit price of one product code on an invoice. An invoice can span several rows, and the same product code can appear on multiple lines.

| Field | Meaning |
|---|---|
| `InvoiceNo` | Invoice number, shared by the lines on the same invoice. |
| `StockCode` | Product identifier. Codes can contain letters, as in `85123A`. |
| `Description` | Text description of the product. |
| `Quantity` | Units on this line. Negative values occur in the raw data. Cleaning distinguishes purchases, returns, and adjustments. |
| `InvoiceDate` | Date and time recorded for the invoice. |
| `UnitPrice` | Price per unit in pounds sterling. |
| `CustomerID` | Customer identifier. Some rows have a missing value. |
| `Country` | Customer’s country or region. |

Filter the original table by customer ID and invoice number. The output retains every line on this invoice so you can check the product, quantity, and unit price for each one.

## 交易明细字段

原表中的一行记录某张发票上一个商品编码的交易数量和单价。同一张发票可以占多行，同一商品编码也可能重复出现。

| 列名 | 含义 |
|---|---|
| `InvoiceNo` | 发票编号。同一张发票上的各行共用这个编号。 |
| `StockCode` | 商品编码，可能包含字母，例如 `85123A`。 |
| `Description` | 商品的文字描述。 |
| `Quantity` | 本行记录的商品件数。原表含负数，清洗时需区分购买与退货等记录。 |
| `InvoiceDate` | 发票记录的日期和时间。 |
| `UnitPrice` | 单件价格，单位为英镑。 |
| `CustomerID` | 客户编号。原表中部分行缺少此值。 |
| `Country` | 客户所在的国家或地区。 |

下面用客户编号和发票编号筛选原表。输出保留该发票的每一行，方便逐行核对商品、数量和单价。

In [9]:
invoice = sales.loc[
    (sales["CustomerID"] == "17850") & (sales["InvoiceNo"] == "536365")
].copy()
print(invoice.to_string(index=False))

InvoiceNo StockCode                         Description  Quantity         InvoiceDate  UnitPrice CustomerID        Country
   536365    85123A  WHITE HANGING HEART T-LIGHT HOLDER         6 2010-12-01 08:26:00       2.55      17850 United Kingdom
   536365     71053                 WHITE METAL LANTERN         6 2010-12-01 08:26:00       3.39      17850 United Kingdom
   536365    84406B      CREAM CUPID HEARTS COAT HANGER         8 2010-12-01 08:26:00       2.75      17850 United Kingdom
   536365    84029G KNITTED UNION FLAG HOT WATER BOTTLE         6 2010-12-01 08:26:00       3.39      17850 United Kingdom
   536365    84029E      RED WOOLLY HOTTIE WHITE HEART.         6 2010-12-01 08:26:00       3.39      17850 United Kingdom
   536365     22752        SET 7 BABUSHKA NESTING BOXES         2 2010-12-01 08:26:00       7.65      17850 United Kingdom
   536365     21730   GLASS STAR FROSTED T-LIGHT HOLDER         6 2010-12-01 08:26:00       4.25      17850 United Kingdom


## Aggregate invoice lines

For this invoice, `len(invoice)` counts lines, `nunique()` counts distinct product codes, and summing `Quantity` gives the number of units purchased. Multiply quantity by unit price on each line before summing the amounts. This preserves price differences across lines for the same product.

## 发票汇总口径

对这张发票，`len(invoice)` 统计明细行数，`nunique()` 统计不同商品编码的数量，`Quantity.sum()` 统计总件数。计算金额时，先把每行的数量乘以该行的单价，再对行金额求和。这样可以保留同一商品在不同明细行上的价格差异。

In [10]:
line_amount = invoice["Quantity"] * invoice["UnitPrice"]
print("Invoice lines / 明细行数：", len(invoice))
print("Distinct products / 不同商品种数：", invoice["StockCode"].nunique())
print("Units purchased / 购买总件数：", invoice["Quantity"].sum())
print(f"First line / 第一行金额：£{line_amount.iloc[0]:.2f}")
print(f"Invoice total / 整张发票金额：£{line_amount.sum():.2f}")

Invoice lines / 明细行数： 7
Distinct products / 不同商品种数： 7
Units purchased / 购买总件数： 40
First line / 第一行金额：£15.30
Invoice total / 整张发票金额：£139.12


## Verified totals and their role in recommendation

The first line records 6 units of product `85123A` at £2.55 each, giving £15.30. The invoice contains 7 lines, 7 distinct products, and 40 units, totalling £139.12.

In a purchase history, one valid purchase invoice for a customer forms one basket. The research plan removes exact duplicate rows and filters for valid purchases before combining lines for the same product on that invoice. Count one purchase event for the product in that basket and sum its quantities and amounts. A later purchase of the same product on another invoice counts as a separate event.

## 核验结果及其在推荐系统中的用途

商品 `85123A` 在第一行出现，6 件 × £2.55 = £15.30。整张发票共 7 行、7 种商品、40 件，总金额为 £139.12。

构建购买历史时，同一位客户的一张有效购买发票构成一个购物篮。按项目研究计划，应先去掉完全重复的原始行并筛出有效购买，再把同一张发票中同一商品的多行合并：商品在这个篮子里记一次购买事件，件数和金额分别累加。同一客户在另一张发票上再次购买它，则保留为另一次购买事件。